In [1]:
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import torch
from IPython.display import *


from pacer import (
    # read_dat_file,
    DatVersion,
    RawGPSSource,
    GPMFSource,
    SequentialGPSSource,
    CoordinateSystem,
    GPSSample,
    Vec3f,
    PointInTime_GPSSample,
    Point,
    Laps,
    Segment,
    Lap,
)

In [2]:
# vector helpers (pacer.Point exposes only x, y, rot -- no operators / scalar)
def vadd(p, q):   return Point(p.x + q.x, p.y + q.y)
def vsub(p, q):   return Point(p.x - q.x, p.y - q.y)
def vscale(s, p): return Point(s * p.x, s * p.y)
def vdot(p, q):   return p.x * q.x + p.y * q.y

In [3]:
files = [
    "/Users/daniil/Desktop/D24/GX010060.MP4",
]

gpmf = GPMFSource(files[0])
# for f in files[1:]:
#     gpmf = SequentialGPSSource(gpmf, GPMFSource(f))

gpmf.get_total_duration()
samples = []


def on_sample(s: GPSSample, _, _2):
    if s.full_speed > 3:
        samples.append((s, gpmf.current_time_span()))


while not gpmf.is_end():
    gpmf.read_samples(on_sample)
    gpmf.next()

In [4]:
s1, s2 = samples[0][0], samples[531][0]
print(f"Sample 1: {s1}")
print(f"Sample 2: {s2}")

Sample 1: GPSSample(lat=52.040054, lon=-0.785116, altitude=70.778000, full_speed=3.290000, ground_speed=4.361000)
Sample 2: GPSSample(lat=52.039415, lon=-0.781324, altitude=73.886000, full_speed=10.550000, ground_speed=10.533000)


In [5]:
cs = CoordinateSystem(s1)

In [6]:
rough_frequency = len(samples) / len(set(span for _, span in samples))

data = np.array(
    [
        cs.distance(s1, s2)
        / (0.5 * s1.full_speed + 0.5 * s2.full_speed)
        * rough_frequency
        for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])
    ]
)

px.scatter(data)

In [7]:
di = np.round(
    np.array(
        [
            cs.distance(s1, s2)
            / (0.5 * s1.full_speed + 0.5 * s2.full_speed)
            * rough_frequency
            for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])
        ]
    )
)

px.scatter(di)

In [8]:
floor = torch.Tensor([b for (_, (b, _)) in samples])
ceil = torch.Tensor([e for (_, (_, e)) in samples])
di = np.round(
    np.array(
        [
            cs.distance(s1, s2)
            / (0.5 * s1.full_speed + 0.5 * s2.full_speed)
            * rough_frequency
            for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])
        ]
    )
)
di = torch.Tensor(np.concatenate([[1], di]))

assert ceil.shape == floor.shape == di.shape


def loss(x):
    assert x.shape == di.shape, f"Expected {di.shape}, got {x.shape}"

    my_diffs = x[1:] - x[:-1]
    my_diffs /= di[1:]
    spacing = ((my_diffs - my_diffs.mean()) ** 2).mean()
    constraints = (((floor - x).clip(min=0) + (x - ceil).clip(min=0)) ** 2).mean()
    return spacing + constraints

In [9]:
t1 = (floor + ceil) / 2
t1.requires_grad_()

learning_curve = []

for lr in [1e-1, 1e-2, 1e-3]:
    optimizer = torch.optim.Adam([t1], lr=lr)
    for i in range(100):
        optimizer.zero_grad()
        l = loss(t1)
        l.backward()
        optimizer.step()
        learning_curve.append((lr, i, l.item()))
px.line(
    pd.DataFrame(learning_curve, columns=["lr", "iteration", "loss"]),
    y="loss",
    log_y=True,
    color="lr",
    title="Learning curve",
)

In [10]:
phase = torch.tensor(floor[0], requires_grad=True)
frequency = torch.tensor(rough_frequency, requires_grad=True)

t2 = phase + 1 / frequency * (di.long().cumsum(0).float() - 1)
learning_curve = []

for lr in [1e-1, 1e-2, 1e-3]:
    optimizer = torch.optim.Adam([phase, frequency], lr=lr)
    for i in range(100):
        optimizer.zero_grad()
        l = loss(t2)
        l.backward(retain_graph=True)
        optimizer.step()
        t2 = phase + 1 / frequency * (di.long().cumsum(0).float() - 1)
        learning_curve.append((lr, i, l.item()))

px.line(
    pd.DataFrame(learning_curve, columns=["lr", "iteration", "loss"]),
    y="loss",
    log_y=True,
    color="lr",
    title="Learning curve",
)

/tmp/claude-501/ipykernel_30734/2896287337.py:1: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [11]:
implied_frequency = di[1:] / (t1[1:] - t1[:-1])
px.scatter(implied_frequency.detach().numpy(), title="Implied frequency from GPS data")

In [12]:
implied_frequency = di[1:] / (t2[1:] - t2[:-1])
px.scatter(implied_frequency.detach().numpy(), title="Implied frequency from GPS data")

In [13]:
laps = Laps()
laps.set_coordinate_system(cs)

for (s, span), t in zip(samples, t1):
    laps.add_point(s, t)

s = laps.pick_random_start()
laps.sectors.start_line = s
laps.update()

pd.DataFrame([dict(lap=i, lap_time=laps.lap_time(i)) for i in range(laps.laps_count())])

,lap,lap_time
0,0,NaN
1,1,NaN
2,2,NaN
3,3,NaN
4,4,NaN
5,5,NaN
6,6,NaN
7,7,NaN
8,8,NaN
9,9,NaN


In [14]:
delta_by_lap = []
for i in range(1, laps.laps_count()):
    reference_lap = laps.get_lap(i)
    reference_lap.width = 5
    lap0 = reference_lap.resample(laps.get_lap(3), cs)
    lap1 = reference_lap.resample(laps.get_lap(4), cs)

    m = min(len(lap0.points), len(lap1.points))
    if m == 0:
        continue
    t0 = np.array([lap0.points[j].time for j in range(m)]) - lap0.points[0].time
    t1 = np.array([lap1.points[j].time for j in range(m)]) - lap1.points[0].time
    delta = t1 - t0

    delta_by_lap.append(
        pd.DataFrame(
            dict(
                distance=list(lap0.cum_distances)[:m],
                delta=delta,
                reference_lap=f"Lap {i}",
            )
        )
    )

delta_by_lap = pd.concat(delta_by_lap)

px.line(
    delta_by_lap,
    x="distance",
    y="delta",
    color="reference_lap",
)

In [15]:
delta_by_lap = []
reference_lap = laps.get_lap(1)

for i in range(1, laps.laps_count() - 1):
    reference_lap.width = 5
    lap0 = reference_lap.resample(laps.get_lap(i), cs)
    lap1 = reference_lap.resample(laps.get_lap(5), cs)

    m = min(len(lap0.points), len(lap1.points))
    if m == 0:
        continue
    t0 = np.array([lap0.points[j].time for j in range(m)]) - lap0.points[0].time
    t1 = np.array([lap1.points[j].time for j in range(m)]) - lap1.points[0].time
    delta = t1 - t0

    delta_by_lap.append(
        pd.DataFrame(
            dict(
                lat=[lap0.points[j].point.lat for j in range(m)],
                lon=[lap0.points[j].point.lon for j in range(m)],
                distance=list(lap0.cum_distances)[:m],
                delta=delta,
                lap=i,
            )
        )
    )

delta_by_lap = pd.concat(delta_by_lap)

px.line(
    delta_by_lap,
    x="distance",
    y="delta",
    color="lap",
)

In [16]:
ddelta = delta_by_lap["delta"].diff()
ddelta = ddelta.clip(
    lower=delta.mean() - 2 * ddelta.std(), upper=ddelta.mean() + 2 * ddelta.std()
).rolling(20).mean()


px.scatter_map(
    delta_by_lap.assign(ddelta=ddelta).loc[lambda d: d["lap"] == 1],
    lat="lat",
    lon="lon",
    color="ddelta",
    zoom=17,
).update_layout(height=800)

In [17]:
px.histogram(delta_by_lap["delta"], nbins=200)

In [18]:
lap = laps.get_lap(1)


def build_lap_df(lap):
    n = min(len(lap.points), len(lap.cum_distances))
    return pd.DataFrame(
        [
            dict(
                lat=lap.points[i].point.lat,
                lon=lap.points[i].point.lon,
                time=lap.points[i].time - lap.points[0].time,
                distance=lap.cum_distances[i],
                i_point=i,
            )
            for i in range(n)
        ]
    )


all_laps = pd.concat(
    [build_lap_df(laps.get_lap(i)).assign(i_lap=i) for i in range(laps.laps_count())]
)
all_laps

,lat,lon,time,distance,i_point,i_lap
0,52.039752,-0.783212,NaN,0.000000,0,0
1,52.039747,-0.783207,NaN,0.696424,1,0
2,52.039734,-0.783191,NaN,2.526712,2,0
3,52.039721,-0.783174,NaN,4.349569,3,0
4,52.039709,-0.783157,NaN,6.175767,4,0
...,...,...,...,...,...,...
707,52.039786,-0.783219,NaN,1103.384535,707,20
708,52.039774,-0.783200,NaN,1105.207797,708,20
709,52.039766,-0.783187,NaN,1106.516253,709,20
0,52.039766,-0.783187,NaN,0.000000,0,21


In [19]:
fig = px.scatter_map(
    all_laps,
    lat="lat",
    lon="lon",
    color="distance",
    hover_data=["i_lap", "i_point"],
    # map_style="basic",
    zoom=17,
)
fig.update_layout(height=800)


In [20]:
start_line = laps.sectors.start_line
p1, p2 = start_line.first, start_line.second

In [21]:
s1, s2 = map(lambda p: cs.global_(Vec3f(p.x, p.y, 0)), (p1, p2))
s1, s2

(GPSSample(lat=52.039730, lon=-0.783255, altitude=70.779374, full_speed=0.000000, ground_speed=0.000000),
 GPSSample(lat=52.039788, lon=-0.783144, altitude=70.779498, full_speed=0.000000, ground_speed=0.000000))

In [22]:
def add_segment(fig: go.Figure, s1: GPSSample, s2: GPSSample, name: str):
    fig.add_trace(
        go.Scattermap(
            mode="lines",
            lon=[s1.lon, s2.lon],
            lat=[s1.lat, s2.lat],
            name=name,
        )
    )
    return fig


add_segment(fig, s1, s2, "start_line")

In [23]:
Point(1, 2)

Point(x=1.000000, y=2.000000)

In [24]:
start_line

Segment(first=(127.393032, -36.055402), second=(135.011597, -29.577944))

In [25]:
def to_point(s: GPSSample):
    v = cs.local(s)
    return Point(v.x, v.y)


first_seg = Segment(
    to_point(laps.get_lap(0).points[0].point), to_point(laps.get_lap(0).points[1].point)
)

add_segment(
    fig, laps.get_lap(0).points[0].point, laps.get_lap(0).points[1].point, "first_one"
)

In [26]:
a, b = first_seg.first, first_seg.second
x, y = start_line.first, start_line.second
nrm = Point(-(x.y - y.y), x.x - y.x)  # perpendicular to the start line
da = vdot(nrm, vsub(a, x))
db = vdot(nrm, vsub(b, x))
ratio = da / (da - db) if (da - db) != 0 else 0.5
ratio

-0.030370282339245953

In [27]:
df = pd.DataFrame(
    dict(
        x=[
            start_line.first.x,
            start_line.second.x,
            None,
            first_seg.first.x,
            first_seg.second.x,
        ],
        y=[
            start_line.first.y,
            start_line.second.y,
            None,
            first_seg.first.y,
            first_seg.second.y,
        ],
        name=[1, 1, None, 2, 3],
    )
)

fig = px.line(df, x="x", y="y", hover_data="name")
fig

In [28]:
x, y = start_line.first, start_line.second
a, b = first_seg.first, first_seg.second

In [29]:
r = ratio if ratio is not None else 0.5
res = vadd(vscale(1 - r, a), vscale(r, b))
fig.add_trace(go.Scatter(x=[res.x], y=[res.y]))

In [30]:
def rot(x):
    return Point(-x.y, x.x)

In [31]:
n = rot(vsub(x, y))
vdot(n, vsub(a, x)), vdot(n, vsub(b, x)), vdot(n, vsub(a, y)), vdot(n, vsub(b, y))

(0.2080851222063984, 7.059688274327929, 0.2080851222063984, 7.059688274327932)

In [32]:
a

Point(x=130.339105, y=-33.577904)

In [33]:
dir(start_line.first)

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'rot',
 'x',
 'y']

In [34]:
vdot(start_line.first, start_line.first)

17528.976724271815

In [35]:
start_line.intersects(first_seg.first, first_seg.second, 0.0)

False

In [36]:
cs.global_(Vec3f(0, 0, 0))

GPSSample(lat=52.040054, lon=-0.785116, altitude=70.778000, full_speed=0.000000, ground_speed=0.000000)

In [37]:
points_df = pd.DataFrame(
    {
        "latitude": [s.lat for s, _ in samples],
        "longitude": [s.lon for s, _ in samples],
        "speed": [s.full_speed for s, _ in samples],
    }
)

px.scatter_map(
    points_df,
    lat="latitude",
    lon="longitude",
    color="speed",
    map_style="outdoors",
).update_layout(height=800)

In [38]:
lap = laps.get_lap(1)
dir(lap)

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'count',
 'cum_distances',
 'fill_distances',
 'lap_time',
 'points',
 'resample',
 'timing_line',
 'timing_lines_count',
 'width']

In [39]:
lap.points

In [40]:
delta_by_lap

,lat,lon,distance,delta,lap
0,52.039759,-0.783200,0.000000,NaN,1
1,52.039769,-0.783291,6.427953,NaN,1
2,52.039785,-0.783381,12.891003,NaN,1
0,52.039782,-0.783157,0.000000,NaN,2
1,52.039764,-0.783190,6.427953,NaN,2
0,52.039764,-0.783190,0.000000,NaN,3
1,52.039799,-0.783280,6.427953,NaN,3
2,52.039752,-0.783213,12.891003,NaN,3
0,52.039752,-0.783213,0.000000,NaN,4
1,52.039763,-0.783191,6.427953,NaN,4


In [41]:
px.scatter(delta[1:] - delta[:-1])

In [42]:
c = 0.1
noise = np.round((delta[1:] - delta[:-1]) / c) * c
px.scatter(noise)

In [43]:
px.scatter(np.cumsum(noise), title="Cumulative noise")

In [44]:
px.line(delta - np.concatenate([[0], np.cumsum(noise)]))

In [45]:
laps = Laps()
laps.set_coordinate_system(cs)

for (s, span), t in zip(samples, t2):
    laps.add_point(s, t)
s = laps.pick_random_start()
laps.sectors.start_line = s
laps.update()
reference_lap = laps.get_lap(1)
reference_lap.width = 5
lap0 = reference_lap.resample(laps.get_lap(3), cs)
lap1 = reference_lap.resample(laps.get_lap(4), cs)
m = min(len(lap0.points), len(lap1.points))
px.line(
    [
        lap1.points[i].time
        - lap0.points[i].time
        - lap1.points[0].time
        + lap0.points[0].time
        for i in range(m)
    ]
)